# Activity 7 – Additional Exercises (NumPy and Pandas)

**Student:** Campos | **Student ID:** TUPM-26-1247

Dataset: UCI SECOM semiconductor manufacturing data (`secom.data`, `secom_labels.data`, `secom.names`).
The SECOM files are read from the `data` folder of the Activity 7 repository and are **not** uploaded to GitHub.

Label convention used here (from `secom.names`): `-1` = **Pass**, `1` = **Fail**. Change `PASS_LABEL` / `FAIL_LABEL` in Cell 1 if your instructor's convention differs.

In [1]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

LAST_NAME = "Campos"
STUDENT_ID = "TUPM-26-1247"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits"

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

# SECOM label convention (secom.names): -1 = Pass, 1 = Fail
PASS_LABEL, FAIL_LABEL = -1, 1

N_BOOT = 200 + 10 * SEED_DIGIT

np.set_printoptions(precision=4, suppress=True, linewidth=120)
pd.options.display.width = 160
pd.options.display.max_columns = 30

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID sum:", ID_SUM)
print("Name length:", NAME_LENGTH)
print("Random seed:", RANDOM_SEED)
print("Verification value:", verification_value)
print("Bootstrap repetitions:", N_BOOT)

Student: Campos TUPM-26-1247
Seed Digit: 7
ID sum: 22
Name length: 6
Random seed: 261247
Verification value: 139
Bootstrap repetitions: 270


In [2]:
#[CELL 2] LOCATE AND LOAD THE SECOM FILES
WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR                      # outputs are saved next to this notebook

candidates = [WORK_DIR / "data", WORK_DIR.parent / "data"]
DATA_DIR = next((p for p in candidates if (p / "secom.data").is_file()), None)
assert DATA_DIR is not None, "secom.data not found in ./data or ../data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"
for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing File: {path}"

raw = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw.ndim == 2
assert len(labels) == raw.shape[0]
assert np.isin(labels, [PASS_LABEL, FAIL_LABEL]).all()

record_ids = np.arange(raw.shape[0])       # original SECOM record identifiers (row order)
is_fail = labels == FAIL_LABEL

print("Workspace folder:", WORK_DIR.name)
print("Data folder:", DATA_DIR.name)
print("Measurement Matrix Shape:", raw.shape)
print("Missing Values:", int(np.isnan(raw).sum()))
print("Pass Records:", int((~is_fail).sum()), "| Fail Records:", int(is_fail.sum()))

Workspace folder: Lab7_NumPy_Pandas_Additional_Exercises
Data folder: data
Measurement Matrix Shape: (1567, 590)
Missing Values: 41951
Pass Records: 1463 | Fail Records: 104


## Exercise 1 – Vectorized Measurement Quality Analysis
NumPy only: Boolean indexing, `nan`-aware statistics and a robust (median/MAD) standardization.
Robust values use the modified z-score `0.6745 × (x − median) / MAD`, flagged when `|value| > 3.5`.
A feature with `MAD = 0` has no robust scale, so its observed values are scored `0` (never flagged) instead of dividing by zero.

In [3]:
#[CELL 3] FEATURE SCREENING (MISSING AND CONSTANT FEATURES)
missing = np.isnan(raw)
observed = (~missing).sum(axis=0)
has_missing = missing.any(axis=0)
all_missing = observed == 0

obs_min = np.where(missing, np.inf, raw).min(axis=0)
obs_max = np.where(missing, -np.inf, raw).max(axis=0)
constant = (observed > 0) & (obs_min == obs_max)

usable_mask = ~(all_missing | constant)
usable_indices = np.flatnonzero(usable_mask)

print("Total Features:", raw.shape[1])
print("Features With Missing Values:", int(has_missing.sum()))
print("Entirely Missing Features:", int(all_missing.sum()))
print("Constant Features:", int(constant.sum()))
print("Usable Features:", len(usable_indices))

Total Features: 590
Features With Missing Values: 538
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [4]:
#[CELL 4] REPRODUCIBLE FEATURE SELECTION: 5 + (SEED_DIGIT % 3)
N_QUALITY = 5 + (SEED_DIGIT % 3)

rng_quality = np.random.default_rng(RANDOM_SEED)
q_idx = np.sort(rng_quality.choice(usable_indices, size=N_QUALITY, replace=False))
q_names = [f"Feature_{i:03d}" for i in q_idx]
Xq = raw[:, q_idx]

print("Features To Select:", N_QUALITY)
print("Selected Features:", q_names)

Features To Select: 6
Selected Features: ['Feature_118', 'Feature_249', 'Feature_271', 'Feature_310', 'Feature_480', 'Feature_556']


In [5]:
#[CELL 5] NUMPY FEATURE STATISTICS
q_observed = (~np.isnan(Xq)).sum(axis=0)
q_mean = np.nanmean(Xq, axis=0)
q_median = np.nanmedian(Xq, axis=0)
q_sd = np.nanstd(Xq, axis=0, ddof=1)
q_missing_pct = 100 * np.isnan(Xq).mean(axis=0)

stats_table = pd.DataFrame(
    {"Observed": q_observed, "Mean": q_mean, "Median": q_median,
     "Sample_SD": q_sd, "Missing_Pct": q_missing_pct},
    index=q_names,
)
print(stats_table.round(4))

             Observed     Mean   Median  Sample_SD  Missing_Pct
Feature_118      1543   0.5984   0.5990     0.0081       1.5316
Feature_249      1567   0.0011   0.0000     0.0158       0.0000
Feature_271      1561  46.0566  45.6765    17.8664       0.3829
Feature_310      1566   0.3048   0.3029     0.0435       0.0638
Feature_480      1561  75.5381  70.4345    35.7525       0.3829
Feature_556      1307   4.2169   4.0671     1.2800      16.5922


In [6]:
#[CELL 6] ROBUST STANDARDIZATION (MEDIAN AND MAD) AND FLAGGING
MAD_SCALE = 0.6745
THRESHOLD = 3.5

mad = np.nanmedian(np.abs(Xq - q_median), axis=0)
zero_mad = mad == 0
safe_mad = np.where(zero_mad, 1.0, mad)

robust_z = MAD_SCALE * (Xq - q_median) / safe_mad
robust_z = np.where(zero_mad & ~np.isnan(Xq), 0.0, robust_z)   # zero MAD: score 0, NaN stays NaN

flagged = np.abs(robust_z) > THRESHOLD
flag_counts = flagged.sum(axis=0)
flag_pct = 100 * flag_counts / q_observed

print("MAD:", mad)
print("Zero-MAD Features:", [n for n, z in zip(q_names, zero_mad) if z])
print("Flagged Per Feature:", flag_counts)
print("Flagged Percentage:", flag_pct)

MAD: [ 0.0046  0.     15.0279  0.0272 19.3867  0.4427]
Zero-MAD Features: ['Feature_249']
Flagged Per Feature: [12  0  1  1 25 12]
Flagged Percentage: [0.7777 0.     0.0641 0.0639 1.6015 0.9181]


In [7]:
#[CELL 7] RANK FEATURES AND EXPORT measurement_quality.csv
quality = pd.DataFrame({
    "Feature": q_names,
    "Observed": q_observed,
    "Missing_Pct": q_missing_pct,
    "Mean": q_mean,
    "Median": q_median,
    "Sample_SD": q_sd,
    "MAD": mad,
    "Zero_MAD": zero_mad,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct,
})
quality = quality.sort_values(["Flagged_Pct", "Flagged_Count"], ascending=False).reset_index(drop=True)
quality.insert(0, "Rank", np.arange(1, len(quality) + 1))
quality.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

print(quality.round(4).to_string(index=False))

 Rank     Feature  Observed  Missing_Pct    Mean  Median  Sample_SD     MAD  Zero_MAD  Flagged_Count  Flagged_Pct
    1 Feature_480      1561       0.3829 75.5381 70.4345    35.7525 19.3867     False             25       1.6015
    2 Feature_556      1307      16.5922  4.2169  4.0671     1.2800  0.4427     False             12       0.9181
    3 Feature_118      1543       1.5316  0.5984  0.5990     0.0081  0.0046     False             12       0.7777
    4 Feature_271      1561       0.3829 46.0566 45.6765    17.8664 15.0279     False              1       0.0641
    5 Feature_310      1566       0.0638  0.3048  0.3029     0.0435  0.0272     False              1       0.0639
    6 Feature_249      1567       0.0000  0.0011  0.0000     0.0158  0.0000      True              0       0.0000


In [8]:
#[CELL 8] VERIFY VECTORIZED RESULTS AGAINST A FEATURE-BY-FEATURE LOOP
loop_ok = True
for j in range(N_QUALITY):
    col = Xq[:, j]
    obs = col[~np.isnan(col)]
    med = np.median(obs)
    m = np.median(np.abs(obs - med))
    n_flag = 0 if m == 0 else int((np.abs(MAD_SCALE * (obs - med) / m) > THRESHOLD).sum())
    loop_ok = loop_ok and (
        n_flag == flag_counts[j]
        and np.isclose(obs.mean(), q_mean[j])
        and np.isclose(obs.std(ddof=1), q_sd[j])
        and np.isclose(med, q_median[j])
    )
vectorized_verified = bool(loop_ok)

hi_missing = int(np.argmax(q_missing_pct))
top_flag_feature = quality.loc[0, "Feature"]
top_flag_pct = float(quality.loc[0, "Flagged_Pct"])
total_flagged = int(flag_counts.sum())

print("Highest Missing Percentage:", f"{q_missing_pct[hi_missing]:.4f}% ({q_names[hi_missing]})")
print("Highest Flagged Feature:", top_flag_feature, f"({top_flag_pct:.4f}%)")
print("Total Flagged Observations:", total_flagged)
print("Vectorized Results Verified:", vectorized_verified)

Highest Missing Percentage: 16.5922% (Feature_556)
Highest Flagged Feature: Feature_480 (1.6015%)
Total Flagged Observations: 51
Vectorized Results Verified: True


## Exercise 2 – Balanced Pass/Fail Profiling with Pandas
Labeled DataFrame indexed by the original SECOM record identifier (`Example_ID`), a reproducible balanced sample,
median imputation (medians from the **full** dataset), `.groupby()` / `.agg()` statistics, and the standardized mean difference (SMD).

The five features are drawn from usable features that have some missing values (so imputation matters) and at least
10 observed measurements in each outcome group. The same five features are reused in Exercise 3.

In [9]:
#[CELL 9] LABELED DATAFRAME WITH ORIGINAL RECORD IDENTIFIERS
feature_names = [f"Feature_{i:03d}" for i in range(raw.shape[1])]
record_index = pd.Index(record_ids, name="Example_ID")

features_df = pd.DataFrame(raw, columns=feature_names, index=record_index)
label_df = pd.DataFrame(
    {"Result": labels, "Outcome": np.where(is_fail, "Fail", "Pass")},
    index=record_index,
)
df = pd.concat([features_df, label_df], axis=1)

assert np.array_equal(df["Result"].to_numpy(), labels)
print("DataFrame Shape:", df.shape)
print("Index Name:", df.index.name, "| First IDs:", df.index[:5].tolist())
print("Outcome Counts:\n", df["Outcome"].value_counts())

DataFrame Shape: (1567, 592)
Index Name: Example_ID | First IDs: [0, 1, 2, 3, 4]
Outcome Counts:
 Outcome
Pass    1463
Fail     104
Name: count, dtype: int64


In [10]:
#[CELL 10] SELECT FIVE USABLE FEATURES (REPRODUCIBLE)
obs_fail = (~missing[is_fail]).sum(axis=0)
obs_pass = (~missing[~is_fail]).sum(axis=0)
eligible = usable_mask & has_missing & (obs_fail >= 10) & (obs_pass >= 10)

rng_select = np.random.default_rng(RANDOM_SEED)
sel_idx = np.sort(rng_select.choice(np.flatnonzero(eligible), size=5, replace=False))
sel_names = [f"Feature_{i:03d}" for i in sel_idx]

print("Eligible Features:", int(eligible.sum()))
print("Selected Features:", sel_names)
print("Observed (Fail):", obs_fail[sel_idx])
print("Observed (Pass):", obs_pass[sel_idx])
print("Missing Values:", missing[:, sel_idx].sum(axis=0))

Eligible Features: 418
Selected Features: ['Feature_109', 'Feature_228', 'Feature_269', 'Feature_472', 'Feature_553']
Observed (Fail): [ 30 104 104 104  86]
Observed (Pass): [ 519 1461 1455 1456 1221]
Missing Values: [1018    2    8    7  260]


In [11]:
#[CELL 11] MEDIAN IMPUTATION USING FULL-DATASET MEDIANS
full_medians = np.nanmedian(raw[:, sel_idx], axis=0)
imputed = np.where(np.isnan(raw[:, sel_idx]), full_medians, raw[:, sel_idx])
imputed_df = pd.DataFrame(imputed, columns=sel_names, index=record_index)

print("Full-Dataset Medians:", full_medians)
print("Missing Before:", int(missing[:, sel_idx].sum()), "| After:", int(np.isnan(imputed).sum()))

Full-Dataset Medians: [  0.981    0.0155   3.7035 138.2552   7.4279]
Missing Before: 1295 | After: 0


In [12]:
#[CELL 12] REPRODUCIBLE BALANCED SAMPLE
n_pass_total = int((~is_fail).sum())
n_fail_total = int(is_fail.sum())
n_per_group = min(20 + ID_SUM, n_pass_total, n_fail_total)

rng_balance = np.random.default_rng(RANDOM_SEED)
pass_pos = rng_balance.choice(np.flatnonzero(~is_fail), size=n_per_group, replace=False)
fail_pos = rng_balance.choice(np.flatnonzero(is_fail), size=n_per_group, replace=False)
balanced_pos = np.sort(np.concatenate([pass_pos, fail_pos]))

balanced = pd.concat([imputed_df, label_df], axis=1).iloc[balanced_pos]
balanced.reset_index().to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)

print("Records Per Outcome:", n_per_group)
print("Balanced Sample Size:", len(balanced))
print(balanced["Outcome"].value_counts())
print(balanced.head())

Records Per Outcome: 42
Balanced Sample Size: 84
Outcome
Fail    42
Pass    42
Name: count, dtype: int64
            Feature_109  Feature_228  Feature_269  Feature_472  Feature_553  Result Outcome
Example_ID                                                                                 
10                0.981       0.0128       3.6163     219.3969       7.8882       1    Fail
14                0.981       0.0123       4.2300     169.5462       6.3017       1    Fail
21                0.981       0.0217       2.3222     151.6133      10.8976      -1    Pass
40                0.981       0.0165       2.9624     161.6201       8.7395       1    Fail
45                0.981       0.0137       2.6861      93.7285       8.2986       1    Fail


In [13]:
#[CELL 13] GROUPED STATISTICS WITH groupby() AND agg()
grouped = balanced.groupby("Outcome")[sel_names]
group_stats = grouped.agg(["count", "mean", "median", "std"])
print(group_stats.round(4).T.to_string())

Outcome                 Fail      Pass
Feature_109 count    42.0000   42.0000
            mean      0.9808    0.9807
            median    0.9810    0.9810
            std       0.0011    0.0013
Feature_228 count    42.0000   42.0000
            mean      0.0177    0.0205
            median    0.0148    0.0187
            std       0.0097    0.0138
Feature_269 count    42.0000   42.0000
            mean      3.5617    3.5964
            median    3.3686    3.6688
            std       1.1060    0.8277
Feature_472 count    42.0000   42.0000
            mean    141.7201  142.5925
            median  137.5093  149.8657
            std      53.8121   46.9886
Feature_553 count    42.0000   42.0000
            mean      7.6044    8.0474
            median    7.4279    7.5570
            std       1.6620    1.3644


In [14]:
#[CELL 14] MEAN DIFFERENCE, STANDARDIZED MEAN DIFFERENCE AND RANKING
g_n = grouped.count()
g_mean = grouped.mean()
g_median = grouped.median()
g_sd = grouped.std(ddof=1)

n_f, n_p = g_n.loc["Fail"], g_n.loc["Pass"]
pooled_sd = np.sqrt(
    ((n_f - 1) * g_sd.loc["Fail"] ** 2 + (n_p - 1) * g_sd.loc["Pass"] ** 2) / (n_f + n_p - 2)
)
mean_diff = g_mean.loc["Fail"] - g_mean.loc["Pass"]
smd = (mean_diff / pooled_sd.where(pooled_sd > 0)).fillna(0.0)    # zero denominator -> SMD 0

feature_comparison = pd.DataFrame({
    "N_Fail": n_f,
    "N_Pass": n_p,
    "Fail_Mean": g_mean.loc["Fail"],
    "Pass_Mean": g_mean.loc["Pass"],
    "Fail_Median": g_median.loc["Fail"],
    "Pass_Median": g_median.loc["Pass"],
    "Fail_SD": g_sd.loc["Fail"],
    "Pass_SD": g_sd.loc["Pass"],
    "Pooled_SD": pooled_sd,
    "Mean_Diff_Fail_minus_Pass": mean_diff,
    "Abs_Mean_Diff": mean_diff.abs(),
    "SMD": smd,
    "Abs_SMD": smd.abs(),
}).rename_axis("Feature").reset_index()

feature_comparison = feature_comparison.sort_values("Abs_SMD", ascending=False).reset_index(drop=True)
feature_comparison.insert(0, "Rank", np.arange(1, len(feature_comparison) + 1))
feature_comparison.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print(feature_comparison[["Rank", "Feature", "Mean_Diff_Fail_minus_Pass", "Pooled_SD", "SMD", "Abs_SMD"]].round(4).to_string(index=False))

 Rank     Feature  Mean_Diff_Fail_minus_Pass  Pooled_SD     SMD  Abs_SMD
    1 Feature_553                    -0.4430     1.5205 -0.2914   0.2914
    2 Feature_228                    -0.0028     0.0119 -0.2349   0.2349
    3 Feature_109                     0.0001     0.0012  0.0848   0.0848
    4 Feature_269                    -0.0348     0.9768 -0.0356   0.0356
    5 Feature_472                    -0.8724    50.5157 -0.0173   0.0173


In [15]:
#[CELL 15] VERIFY PANDAS STATISTICS AGAINST NUMPY
fail_vals = balanced.loc[balanced["Outcome"] == "Fail", sel_names].to_numpy()
pass_vals = balanced.loc[balanced["Outcome"] == "Pass", sel_names].to_numpy()

np_mean_diff = fail_vals.mean(axis=0) - pass_vals.mean(axis=0)
np_pooled = np.sqrt(
    ((len(fail_vals) - 1) * fail_vals.var(axis=0, ddof=1)
     + (len(pass_vals) - 1) * pass_vals.var(axis=0, ddof=1))
    / (len(fail_vals) + len(pass_vals) - 2)
)
np_smd = np.where(np_pooled > 0, np_mean_diff / np.where(np_pooled > 0, np_pooled, 1.0), 0.0)

numpy_pandas_match = bool(
    np.allclose(np_mean_diff, mean_diff.reindex(sel_names).to_numpy())
    and np.allclose(np_smd, smd.reindex(sel_names).to_numpy())
)

top2 = feature_comparison.iloc[0]
print("Highest-Ranked Feature:", top2["Feature"])
print("Absolute Standardized Mean Difference:", f"{top2['Abs_SMD']:.4f}")
print("NumPy-Pandas Statistics Match:", numpy_pandas_match)

Highest-Ranked Feature: Feature_553
Absolute Standardized Mean Difference: 0.2914
NumPy-Pandas Statistics Match: True


## Exercise 3 – Statistical Robustness and Sensitivity Analysis
Two preparation strategies on the **full** dataset, for the same five features:

* **Available-case (AC):** each feature uses only its recorded values.
* **Median imputation (MI):** gaps are filled with the full-dataset median.

Stratified bootstrap: `200 + 10 × SEED_DIGIT` repetitions, resampling with replacement independently inside the Fail group and the Pass group.
The same resampled records are used for both strategies in every repetition, so the strategies are compared on identical samples.
The 95% interval is the 2.5th–97.5th percentile of the bootstrap Fail − Pass mean differences.
Features are ranked by `|mean difference| / pooled within-group SD` under each strategy.

In [16]:
#[CELL 16] STRATEGY DATA AND HELPER FUNCTIONS
X_ac = raw[:, sel_idx]            # available-case data (NaN kept)
X_mi = imputed                    # median-imputed data (full-dataset medians)
fail_rows = np.flatnonzero(is_fail)
pass_rows = np.flatnonzero(~is_fail)

assert (obs_fail[sel_idx] >= 10).all() and (obs_pass[sel_idx] >= 10).all()


def pooled_sd_nan(a, b):
    na = (~np.isnan(a)).sum(axis=0)
    nb = (~np.isnan(b)).sum(axis=0)
    va = np.nanvar(a, axis=0, ddof=1)
    vb = np.nanvar(b, axis=0, ddof=1)
    return np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))


def point_summary(fail_vals, pass_vals):
    diff = np.nanmean(fail_vals, axis=0) - np.nanmean(pass_vals, axis=0)
    sd = pooled_sd_nan(fail_vals, pass_vals)
    abs_std = np.where(sd > 0, np.abs(diff) / np.where(sd > 0, sd, 1.0), 0.0)
    return diff, abs_std


def rank_desc(values):
    return (-values).argsort(kind="stable").argsort() + 1


def bootstrap_mean_differences(seed, n_boot):
    rng = np.random.default_rng(seed)
    f_draw = rng.integers(0, len(fail_rows), size=(n_boot, len(fail_rows)))
    p_draw = rng.integers(0, len(pass_rows), size=(n_boot, len(pass_rows)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        d_ac = np.nanmean(X_ac[fail_rows][f_draw], axis=1) - np.nanmean(X_ac[pass_rows][p_draw], axis=1)
    d_mi = X_mi[fail_rows][f_draw].mean(axis=1) - X_mi[pass_rows][p_draw].mean(axis=1)
    return d_ac, d_mi


print("Selected Features:", sel_names)
print("Bootstrap Repetitions:", N_BOOT)
print("Fail Records:", len(fail_rows), "| Pass Records:", len(pass_rows))

Selected Features: ['Feature_109', 'Feature_228', 'Feature_269', 'Feature_472', 'Feature_553']
Bootstrap Repetitions: 270
Fail Records: 104 | Pass Records: 1463


In [17]:
#[CELL 17] POINT ESTIMATES AND STRATIFIED BOOTSTRAP
diff_ac, std_ac = point_summary(X_ac[fail_rows], X_ac[pass_rows])
diff_mi, std_mi = point_summary(X_mi[fail_rows], X_mi[pass_rows])
rank_ac, rank_mi = rank_desc(std_ac), rank_desc(std_mi)

boot_ac, boot_mi = bootstrap_mean_differences(RANDOM_SEED, N_BOOT)
undefined_reps = int(np.isnan(boot_ac).sum() + np.isnan(boot_mi).sum())

boot_mean_ac, boot_mean_mi = np.nanmean(boot_ac, axis=0), np.nanmean(boot_mi, axis=0)
ci_ac = np.nanpercentile(boot_ac, [2.5, 97.5], axis=0)
ci_mi = np.nanpercentile(boot_mi, [2.5, 97.5], axis=0)

print("Undefined Bootstrap Values:", undefined_reps)
print("Mean Difference (AC):", diff_ac)
print("Mean Difference (MI):", diff_mi)
print("Bootstrap Mean (AC):", boot_mean_ac)
print("Bootstrap Mean (MI):", boot_mean_mi)
print("95% CI Lower (AC):", ci_ac[0])
print("95% CI Upper (AC):", ci_ac[1])
print("95% CI Lower (MI):", ci_mi[0])
print("95% CI Upper (MI):", ci_mi[1])

Undefined Bootstrap Values: 0
Mean Difference (AC): [-0.0003  0.0005  0.0833  0.2331  0.4714]
Mean Difference (MI): [-0.      0.0005  0.0836  0.2313  0.3879]
Bootstrap Mean (AC): [-0.0003  0.0004  0.08    0.2871  0.4808]
Bootstrap Mean (MI): [-0.      0.0005  0.0804  0.2851  0.3972]
95% CI Lower (AC): [ -0.0014  -0.0012  -0.1511 -10.5431  -0.1485]
95% CI Upper (AC): [ 0.0009  0.0024  0.3229 10.3433  1.3387]
95% CI Lower (MI): [ -0.0004  -0.0012  -0.1508 -10.5428  -0.1278]
95% CI Upper (MI): [ 0.0004  0.0024  0.3232 10.3402  1.0884]


In [18]:
#[CELL 18] COMPARE DIRECTION, INTERVALS AND RANKINGS ACROSS STRATEGIES
excl_ac = (ci_ac[0] > 0) | (ci_ac[1] < 0)
excl_mi = (ci_mi[0] > 0) | (ci_mi[1] < 0)
change = diff_mi - diff_ac

robustness = pd.DataFrame({
    "Feature": sel_names,
    "AC_Obs_Fail": obs_fail[sel_idx],
    "AC_Obs_Pass": obs_pass[sel_idx],
    "AC_Mean_Diff": diff_ac,
    "AC_Boot_Mean_Diff": boot_mean_ac,
    "AC_CI_Lower": ci_ac[0],
    "AC_CI_Upper": ci_ac[1],
    "AC_CI_Excludes_Zero": excl_ac,
    "AC_Abs_Std_Diff": std_ac,
    "AC_Rank": rank_ac,
    "MI_Mean_Diff": diff_mi,
    "MI_Boot_Mean_Diff": boot_mean_mi,
    "MI_CI_Lower": ci_mi[0],
    "MI_CI_Upper": ci_mi[1],
    "MI_CI_Excludes_Zero": excl_mi,
    "MI_Abs_Std_Diff": std_mi,
    "MI_Rank": rank_mi,
    "Change_In_Mean_Diff": change,
    "Abs_Change": np.abs(change),
    "Sign_Changed": np.sign(diff_ac) != np.sign(diff_mi),
    "Rank_Changed": rank_ac != rank_mi,
    "CI_Overlap": (ci_ac[0] <= ci_mi[1]) & (ci_mi[0] <= ci_ac[1]),
    "CI_Conclusion_Changed": excl_ac != excl_mi,
})
robustness.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

sign_changed = [n for n, c in zip(sel_names, robustness["Sign_Changed"]) if c]
rank_changed = [n for n, c in zip(sel_names, robustness["Rank_Changed"]) if c]
conclusion_changed = [n for n, c in zip(sel_names, robustness["CI_Conclusion_Changed"]) if c]

k = int(np.argmax(np.abs(change)))
largest_change_feature = sel_names[k]
largest_change = float(change[k])
largest_change_rel = (100 * abs(change[k]) / abs(diff_ac[k])) if diff_ac[k] != 0 else float("nan")

print(robustness[["Feature", "AC_Mean_Diff", "MI_Mean_Diff", "Change_In_Mean_Diff",
                  "AC_Rank", "MI_Rank", "Sign_Changed", "Rank_Changed"]].round(4).to_string(index=False))
print("Features With Changed Difference Sign:", len(sign_changed), sign_changed)
print("Features With Changed Ranking:", len(rank_changed), rank_changed)
print("Features With Changed CI Conclusion:", len(conclusion_changed), conclusion_changed)
print("Largest Change in Mean Difference:", largest_change_feature, f"{largest_change:.4f}",
      f"({largest_change_rel:.2f}% of the available-case difference)")

    Feature  AC_Mean_Diff  MI_Mean_Diff  Change_In_Mean_Diff  AC_Rank  MI_Rank  Sign_Changed  Rank_Changed
Feature_109       -0.0003       -0.0000               0.0002        4        5         False          True
Feature_228        0.0005        0.0005               0.0000        3        3         False         False
Feature_269        0.0833        0.0836               0.0004        2        2         False         False
Feature_472        0.2331        0.2313              -0.0018        5        4         False          True
Feature_553        0.4714        0.3879              -0.0835        1        1         False         False
Features With Changed Difference Sign: 0 []
Features With Changed Ranking: 2 ['Feature_109', 'Feature_472']
Features With Changed CI Conclusion: 0 []
Largest Change in Mean Difference: Feature_553 -0.0835 (17.71% of the available-case difference)


In [19]:
#[CELL 19] REPRODUCIBILITY CHECK (SAME SEED)
boot_ac_2, boot_mi_2 = bootstrap_mean_differences(RANDOM_SEED, N_BOOT)
reproducible = bool(
    np.array_equal(boot_ac, boot_ac_2, equal_nan=True)
    and np.array_equal(boot_mi, boot_mi_2, equal_nan=True)
)

rng_other = bootstrap_mean_differences(RANDOM_SEED + 1, N_BOOT)
differs_with_other_seed = not np.array_equal(boot_ac, rng_other[0], equal_nan=True)

print("Same-Seed Bootstrap Identical:", reproducible)
print("Different Seed Changes Results:", differs_with_other_seed)

Same-Seed Bootstrap Identical: True
Different Seed Changes Results: True


In [20]:
#[CELL 20] EXPORT sensitivity_report.txt
n_boot_total = N_BOOT
lines = [
    "STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Bootstrap Repetitions: {n_boot_total}",
    f"Selected Features: {', '.join(sel_names)}",
    f"Fail Records: {len(fail_rows)} | Pass Records: {len(pass_rows)}",
    "",
    "Mean Difference (Fail - Pass), 95% Percentile CI:",
]
for i, name in enumerate(sel_names):
    lines.append(
        f"- {name}: AC {diff_ac[i]:.4f} [{ci_ac[0][i]:.4f}, {ci_ac[1][i]:.4f}] rank {rank_ac[i]} | "
        f"MI {diff_mi[i]:.4f} [{ci_mi[0][i]:.4f}, {ci_mi[1][i]:.4f}] rank {rank_mi[i]}"
    )
lines += [
    "",
    f"Features With Changed Difference Sign: {len(sign_changed)} {sign_changed}",
    f"Features With Changed Ranking: {len(rank_changed)} {rank_changed}",
    f"Features With Changed CI Conclusion (excludes zero): {len(conclusion_changed)} {conclusion_changed}",
    f"Largest Change in Mean Difference: {largest_change_feature} ({largest_change:.4f}, {largest_change_rel:.2f}% of the available-case difference)",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Interpretation Notes:",
    "- Median imputation fills gaps with the full-dataset median, which pulls both group means toward the overall centre; available-case analysis uses recorded values only.",
    "- Bootstrap intervals describe sampling variability of the mean difference; they do not identify a cause of failure.",
    f"- The Fail group has only {len(fail_rows)} records, so its mean (and the interval) is much less stable than the Pass group's.",
    "- A feature whose direction, interval conclusion or rank changes between strategies is sensitive to how missing values are handled and should be treated with caution.",
]
(OUTPUT_DIR / "sensitivity_report.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
Student: Campos
Student ID: TUPM-26-1247
Random Seed: 261247
Bootstrap Repetitions: 270
Selected Features: Feature_109, Feature_228, Feature_269, Feature_472, Feature_553
Fail Records: 104 | Pass Records: 1463

Mean Difference (Fail - Pass), 95% Percentile CI:
- Feature_109: AC -0.0003 [-0.0014, 0.0009] rank 4 | MI -0.0000 [-0.0004, 0.0004] rank 5
- Feature_228: AC 0.0005 [-0.0012, 0.0024] rank 3 | MI 0.0005 [-0.0012, 0.0024] rank 3
- Feature_269: AC 0.0833 [-0.1511, 0.3229] rank 2 | MI 0.0836 [-0.1508, 0.3232] rank 2
- Feature_472: AC 0.2331 [-10.5431, 10.3433] rank 5 | MI 0.2313 [-10.5428, 10.3402] rank 4
- Feature_553: AC 0.4714 [-0.1485, 1.3387] rank 1 | MI 0.3879 [-0.1278, 1.0884] rank 1

Features With Changed Difference Sign: 0 []
Features With Changed Ranking: 2 ['Feature_109', 'Feature_472']
Features With Changed CI Conclusion (excludes zero): 0 []
Largest Change in Mean Difference: Feature_553 (-0.0835, 17.71% of the available-ca

In [21]:
#[CELL 21] EXPORT VERIFICATION
loaded_quality = pd.read_csv(OUTPUT_DIR / "measurement_quality.csv")
loaded_balanced = pd.read_csv(OUTPUT_DIR / "balanced_sample.csv")
loaded_comparison = pd.read_csv(OUTPUT_DIR / "feature_comparison.csv")
loaded_robust = pd.read_csv(OUTPUT_DIR / "robustness_comparison.csv")

pd.testing.assert_frame_equal(loaded_quality, quality, check_dtype=False, rtol=1e-10, atol=1e-10)
pd.testing.assert_frame_equal(loaded_comparison, feature_comparison, check_dtype=False, rtol=1e-10, atol=1e-10)
pd.testing.assert_frame_equal(loaded_robust, robustness, check_dtype=False, rtol=1e-10, atol=1e-10)

# Balanced sample: values and labels must match the original records by Example_ID
ids = loaded_balanced["Example_ID"].to_numpy()
np.testing.assert_allclose(loaded_balanced[sel_names].to_numpy(), imputed[ids], rtol=1e-10, atol=1e-10)
assert np.array_equal(loaded_balanced["Result"].to_numpy(), labels[ids])
assert (loaded_balanced["Outcome"].value_counts() == n_per_group).all()

for name in ["measurement_quality.csv", "balanced_sample.csv", "feature_comparison.csv",
             "robustness_comparison.csv", "sensitivity_report.txt"]:
    print(name, "Created:", (OUTPUT_DIR / name).is_file())
print("Export Verification: PASSED")

measurement_quality.csv Created: True
balanced_sample.csv Created: True
feature_comparison.csv Created: True
robustness_comparison.csv Created: True
sensitivity_report.txt Created: True
Export Verification: PASSED


In [22]:
#[CELL 22] ASSESSMENT DATA SUMMARY
print("EXERCISE 1 - Vectorized Measurement Quality Analysis")
print("  Selected Features:", q_names)
print("  Highest Missing Percentage:", f"{q_missing_pct[hi_missing]:.4f}% ({q_names[hi_missing]})")
print("  Feature with Highest Flagged Percentage:", top_flag_feature, f"({top_flag_pct:.4f}%)")
print("  Total Flagged Observations:", total_flagged)
print("  Vectorized Results Verified:", vectorized_verified)

print("EXERCISE 2 - Balanced Pass/Fail Profiling")
print("  Balanced Sample Size:", len(balanced), f"({n_per_group} per outcome)")
print("  Selected Features:", sel_names)
print("  Highest-Ranked Feature:", top2["Feature"])
print("  Absolute Standardized Mean Difference:", f"{top2['Abs_SMD']:.4f}")
print("  NumPy-Pandas Statistics Match:", numpy_pandas_match)

print("EXERCISE 3 - Robustness and Sensitivity")
print("  Selected Features:", sel_names)
print("  Bootstrap Repetitions:", N_BOOT)
print("  Largest Change in Mean Difference:", largest_change_feature, f"{largest_change:.4f}")
print("  Features with Changed Difference Sign:", len(sign_changed), sign_changed)
print("  Features with Changed Ranking:", len(rank_changed), rank_changed)
print("  Reproducibility Verified:", reproducible)

EXERCISE 1 - Vectorized Measurement Quality Analysis
  Selected Features: ['Feature_118', 'Feature_249', 'Feature_271', 'Feature_310', 'Feature_480', 'Feature_556']
  Highest Missing Percentage: 16.5922% (Feature_556)
  Feature with Highest Flagged Percentage: Feature_480 (1.6015%)
  Total Flagged Observations: 51
  Vectorized Results Verified: True
EXERCISE 2 - Balanced Pass/Fail Profiling
  Balanced Sample Size: 84 (42 per outcome)
  Selected Features: ['Feature_109', 'Feature_228', 'Feature_269', 'Feature_472', 'Feature_553']
  Highest-Ranked Feature: Feature_553
  Absolute Standardized Mean Difference: 0.2914
  NumPy-Pandas Statistics Match: True
EXERCISE 3 - Robustness and Sensitivity
  Selected Features: ['Feature_109', 'Feature_228', 'Feature_269', 'Feature_472', 'Feature_553']
  Bootstrap Repetitions: 270
  Largest Change in Mean Difference: Feature_553 -0.0835
  Features with Changed Difference Sign: 0 []
  Features with Changed Ranking: 2 ['Feature_109', 'Feature_472']
  Repr